# Smoke Test: WAKFU/en + ANK Superconsolidation + Dashboard

This notebook runs a minimal end-to-end smoke test for the newly added provenance features:

1. Run pipeline for `WAKFU/en` and produce consolidated report artifacts.
2. Build `ANK/en` superconsolidated provenance report from available source reports.
3. Generate a self-contained interactive HTML dashboard for the ANK report.
4. Validate output files exist and print key paths.

In [1]:
from pathlib import Path
import json

# Resolve TB2dic project root from current working directory.
cwd = Path.cwd().resolve()
if (cwd / "src").exists() and (cwd / "testings").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists() and cwd.name.lower() == "testings":
    PROJECT_ROOT = cwd.parent
elif (cwd / "TB2dic" / "src").exists():
    PROJECT_ROOT = cwd / "TB2dic"
else:
    raise RuntimeError(f"Could not resolve TB2dic root from cwd={cwd}")

WORK_DIR = PROJECT_ROOT / "testings" / "smoke_wakfu_en"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK_DIR:", WORK_DIR)

PROJECT_ROOT: C:\Users\Nelso\Documents\MundoDoce\TB2dic
WORK_DIR: C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\smoke_wakfu_en


In [2]:
import os
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure relative paths in params (e.g., dics/...) resolve from project root.
os.chdir(PROJECT_ROOT)

from src.tb2dic import run_pipeline_single
from src.provenance import (
    build_ank_superconsolidated_provenance_report,
    generate_consolidated_report_dashboard,
)


In [5]:

# Step 1: WAKFU/en pipeline smoke run (includes consolidated provenance report generation).
pair_result = run_pipeline_single(
    language="es",
    game="WAKFU",
    output_folder=str(WORK_DIR),
    workers=4,
    batch_size=100,
    provenance_level="detailed",
    provenance_formats=["jsonl", "csv"],
    cleanup_stale=True,
    strict_mode=False,
)

print("pair status:", pair_result.get("status"))
print("pair language:", pair_result.get("language"))
print("pair game:", pair_result.get("game"))
print("pair consolidated jsonl:", (pair_result.get("artifacts") or {}).get("token_provenance_jsonl"))
print("pair consolidated csv:", (pair_result.get("artifacts") or {}).get("token_provenance_csv"))

if pair_result.get("status") != "ok":
    raise RuntimeError(f"WAKFU/en smoke pipeline failed: {pair_result.get('error')}")

BATCH PIPELINE (STEPS 1-5)
Games     : ['WAKFU']
Languages : ['es']
Work dir  : C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\smoke_wakfu_en
Options   : sample=0, workers=4, batch_size=100, add_verb_flags=False, quorum=0.5, cleanup_stale=True, strict_mode=False, pair_workers=1, skip_step_corpusforms=False, provenance_level=detailed, provenance_formats=['jsonl', 'csv']
Dictionary preflight:
  ✅ es: dics\es_dic\es\es_ES.dic

--------------------------------------------------------------------------------
▶ Processing pair: game=WAKFU | language=es
--------------------------------------------------------------------------------
Prewarming filtered i18n for WAKFU/es...
  [corpus] Lingua detector  : target=ES  fallbacks=['FRENCH', 'ENGLISH', 'PORTUGUESE', 'GERMAN']  [built]
  [corpus] Thresholds       : short(<7w)=compare vs EN/FR  long(>=7w)=0.12
  [corpus] Loading WAKFU/es: i18n/WAKFU/texts_es.properties
  [corpus]   -> lang-filtered : 3,312 removed
  [corpus]   -> written       : pr

In [8]:
import os
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure relative paths in params (e.g., dics/...) resolve from project root.
os.chdir(PROJECT_ROOT)

from src.tb2dic import run_pipeline_single
from src.provenance import (
    build_ank_superconsolidated_provenance_report,
    generate_consolidated_report_dashboard,
)

# Step 2: Build ANK superconsolidated report for en from all available source reports in WORK_DIR.
ank_report = build_ank_superconsolidated_provenance_report(
    work_dir=str(WORK_DIR),
    games="all",
    languages=["es"],
    output_formats=["jsonl", "csv"],
    strict_mode=False,
)

print("ANK summary:", json.dumps(ank_report.get("summary", {}), indent=2))

runs = ank_report.get("runs") or []
if not runs:
    raise RuntimeError("No ANK run entries produced")

ank_run_en = runs[0]
print("ANK/en run status:", ank_run_en.get("status"))
if ank_run_en.get("status") != "ok":
    raise RuntimeError(f"ANK/en superconsolidation failed: {ank_run_en.get('error')}")

ank_jsonl = (ank_run_en.get("artifacts") or {}).get("token_provenance_jsonl")
ank_csv = (ank_run_en.get("artifacts") or {}).get("token_provenance_csv")

print("ANK/en consolidated jsonl:", ank_jsonl)
print("ANK/en consolidated csv:", ank_csv)

report_for_dashboard = ank_jsonl or ank_csv
if not report_for_dashboard:
    raise RuntimeError("No ANK consolidated report file available for dashboard generation")

ANK summary: {
  "status": "ok",
  "work_dir": "C:\\Users\\Nelso\\Documents\\MundoDoce\\TB2dic\\testings\\smoke_wakfu_en",
  "selected_games": [
    "WAKFU"
  ],
  "selected_languages": [
    "es"
  ],
  "processed_languages": 1,
  "ok_languages": 1,
  "skipped_languages": 0,
  "failed_languages": 0
}
ANK/en run status: ok
ANK/en consolidated jsonl: C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\smoke_wakfu_en\ANK_es_token_provenance_report.jsonl
ANK/en consolidated csv: C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\smoke_wakfu_en\ANK_es_token_provenance_report.csv


In [10]:
# Step 3: Generate interactive HTML dashboard from ANK/en report.
dashboard_path = str(WORK_DIR / "ANK_es_token_provenance_dashboard.html")

dash_result = generate_consolidated_report_dashboard(
    report_path=r"C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\smoke_wakfu_en\ANK_es_token_provenance_report.jsonl",
    output_html_path=dashboard_path,
    dashboard_title="Smoke Test Dashboard - ANK/en",
)

print("dashboard result:", json.dumps(dash_result, indent=2))


dashboard result: {
  "status": "ok",
  "source_path": "C:\\Users\\Nelso\\Documents\\MundoDoce\\TB2dic\\testings\\smoke_wakfu_en\\ANK_es_token_provenance_report.jsonl",
  "source_format": "jsonl",
  "output_html_path": "C:\\Users\\Nelso\\Documents\\MundoDoce\\TB2dic\\testings\\smoke_wakfu_en\\ANK_es_token_provenance_dashboard.html",
  "rows": 6599
}
